# Family Activity Deep Agent — Shared Calendar & Reminder Drafts

A hands-on tutorial of the **Deep Agents** framework with a standalone **MCP tool server** and durable **SQLite** calendar state.

I built a family coordinator that creates, reads, updates, deletes, and restores activities; detects child and parent conflicts; and stores reviewable reminder drafts.

> **Messaging.** You can set up text or WhatsApp to send these reminders; currently the project only drafts messages. Every calendar or reminder mutation pauses for parent approval.

This notebook follows a hands-on tutorial structure in the same spirit as a GTM-style Deep Agent walkthrough, adapted to the family-calendar use case and this project's real implementation.

## Architecture at a glance

```
Parent request → Family Coordinator (plans, routes, verifies)
                         │
          ┌──────────────┼─────────────────────┐
          ▼              ▼                     ▼
     Calendar work   Weekly planning      Outing research
     + reminders     + review             + cited evidence
          └──────────────┴──────────┬──────────┘
                                ▼
                      Family Activity MCP Server
                                ▼
                  SQLite: events | reminders | audit_logs
```

| Deep Agents concept | Family activity implementation |
|---|---|
| Planning | `write_todos` breaks a parent request into steps |
| Delegation | `task` sends focused work to eight specialists |
| Shared work | Temporary files coordinate one run |
| Tools | 18 local MCP tools; when configured, 7 hosted You.com tools are discovered |
| Skills | Calendar policy, reminder policy, and family preferences load on demand |
| Durable state | SQLite stores events, assignments, reminders, and audit history |
| Human-in-the-loop | LangGraph interrupts before every mutation |


![Family Activity Deep Agent architecture](./images/01_img.png)


## 1. Install and API keys

Run this notebook from the repository root with the project virtual environment. Dependencies are managed by `pyproject.toml`. Never print API keys in notebook output.

In [ ]:
# If needed, install the project into the active notebook kernel:
# %pip install -e '.[dev]'


In [ ]:
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv

load_dotenv()
if not os.getenv("NEBIUS_API_KEY"):
    raise RuntimeError("Add NEBIUS_API_KEY to .env before running the live agent cells.")

print("You.com outing search:", "enabled" if os.getenv("YDC_API_KEY") else "disabled")

if os.getenv("LANGSMITH_API_KEY", "").strip():
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "family-activity-agent-mvp")
    print("LangSmith tracing: ENABLED")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: disabled")

print("Model:", os.getenv("FAMILY_ACTIVITY_MODEL", "nvidia/Nemotron-3-Nano-Omni"))


## 2. Family grounding

The GTM tutorial grounds its agent in company facts. Here the equivalent grounding is the family's operating context: Pacific Time and school hours. Those facts live in a skill rather than being scattered across user prompts.

In [ ]:
preferences = Path("skills/family-preferences/SKILL.md")
print(preferences.read_text())


## 3. MCP tools

The Deep Agent launches the local MCP server as a managed stdio subprocess. Tool definitions live in `src/family_activity_mcp/server.py`; validation and SQLite transactions live in `src/family_activity_mcp/repository.py`.

The LLM decides **which** operation is needed. Plain Python enforces facts the model must not improvise: future times, optimistic versions, family scope, idempotency, and overlap rules.

![MCP tools](./images/02_img.png)


In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from family_activity_agent.agent import mcp_connection

async def discover_tools():
    client = MultiServerMCPClient(mcp_connection())
    return await client.get_tools()

tools = asyncio.run(discover_tools())
for tool in tools:
    print(f"{tool.name}: {tool.description}")


## 4. Skills (progressive disclosure)

The coordinator sees skill descriptions first and loads full `SKILL.md` instructions only when relevant. This keeps the always-on prompt smaller while preserving domain policy.

![Skills progressive disclosure](./images/03_img.png)


In [ ]:
for path in sorted(Path("skills").glob("*/SKILL.md")):
    print(f"\n===== {path} =====")
    print(path.read_text())


## 5. Persistence: database + per-run state

The reference tutorial separates files, long-term memory, and a checkpointer. This project makes a deliberate variation:

- **SQLite** is the durable source of truth across CLI sessions.
- **Temporary files** let agents coordinate during one request and are cleared before the next.
- **`InMemorySaver`** holds graph state required to pause and resume an approval in the current process.

No conversational memory is required to remember calendar facts because the tools read them from the database.

![Persistence boundary](./images/04_img.png)


In [ ]:
from family_activity_agent.cli import RUN_ARTIFACTS

print("Durable state: data/family_activity.db")
print("Ephemeral per-run artifacts:")
for artifact in RUN_ARTIFACTS:
    print(" -", artifact)


## 6. Specialist subagents

Each specialist has a focused prompt and restricted tool set. The coordinator handles simple requests directly to reduce latency and delegates only when specialization adds value.

![Coordinator and specialist agents](./images/05_img.png)


In [ ]:
from family_activity_agent.prompts import (
    INTAKE_PROMPT, CALENDAR_PROMPT, CONFLICT_PROMPT, OUTING_PROMPT, REMINDER_PROMPT
)

subagents = {
    "intake-agent": "Normalizes ambiguous or multi-activity requests",
    "calendar-agent": "Handles complex event lifecycle operations",
    "conflict-agent": "Explains overlaps without mutating state",
    "weekly-planner": "Builds and revises coordinated weekly plans",
    "transportation-agent": "Ranks assignment and transportation candidates",
    "family-outing-agent": "Finds cited ideas for free weekends and school breaks",
    "schedule-reviewer": "Independently audits the full weekly proposal",
    "reminder-agent": "Creates and manages reminder drafts",
}
for name, purpose in subagents.items():
    print(f"{name}: {purpose}")


## 7. Assemble the Deep Agent

This is the core Deep Agents lesson: configure the model, prompt, tools, subagents, skills, backend, checkpointer, and interrupt policy. The framework supplies planning, delegation, filesystem tools, and interrupt plumbing.

`build_family_agent()` also discovers MCP tools and normalizes their results for model tool-message compatibility.

![Assembling the Deep Agent](./images/06_img.png)


In [ ]:
from family_activity_agent.agent import build_family_agent

agent = asyncio.run(build_family_agent())
print("Deep Agent assembled:", agent.name)


## 8. Run one family request

We pass one request and let the coordinator plan, call MCP tools, and pause before the write. `recursion_limit` is the hard backstop against runaway loops.

The date is generated seven days ahead so the demonstration never accidentally creates a past event.

![Run one family request](./images/07_img.png)


In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

demo_day = (datetime.now(ZoneInfo("America/Los_Angeles")) + timedelta(days=7)).date()
run_config = {
    "configurable": {"thread_id": f"notebook-demo-{uuid4()}"},
    "recursion_limit": 30,
    "run_name": "family-activity-notebook-demo",
    "tags": ["family-activity-agent", "notebook", "demo"],
}

request = (
    f"Add Demo Child's soccer practice on {demo_day.isoformat()} "
    "from 4 to 5 PM for family-1"
)
result = asyncio.run(agent.ainvoke(
    {"messages": [{"role": "user", "content": request}]},
    config=run_config,
))
print("Paused for approval:", "__interrupt__" in result)


In [ ]:
# Inspect the visible coordinator messages and its current TODO plan.
for message in result.get("messages", []):
    message.pretty_print()

print("\n===== TODOS =====")
for todo in result.get("todos", []):
    print(f"[{todo.get('status')}] {todo.get('content')}")


![Shared state produced by the run](./images/08_img.png)


In [ ]:
# Inspect temporary shared artifacts created during this run.
for relative_path in [
    "work/event_request.json",
    "work/calendar_plan.json",
    "work/reminder_plan.json",
    "work/weekly_schedule.json",
    "work/assignment_proposal.json",
    "work/outing_proposal.json",
    "reviews/conflict_report.json",
    "reviews/weekly_schedule_review.json",
    "final/completed_action.json",
]:
    path = Path(relative_path)
    if path.exists():
        print(f"\n===== /{relative_path} =====\n{path.read_text()}")


## 9. Human-in-the-loop: approve the calendar gate

`interrupt_on` pauses the graph **before** `create_event`. Inspect the exact tool and arguments, then approve, reject with feedback, or edit the arguments. This mirrors the publish gate in the reference project, but protects calendar mutations.

![Human approval gate](./images/09_img.png)


In [ ]:
from family_activity_agent.cli import pending_requests

requests = pending_requests(result)
for pending in requests:
    print("PENDING APPROVAL:", pending)


### Approve or reject explicitly

Run the approval cell only after reviewing the request above. To reject, replace the decision with `{"type": "reject", "message": "Reason"}`.

In [ ]:
from langgraph.types import Command

if not requests:
    print("No mutation is waiting for approval.")
else:
    decisions = [{"type": "approve"} for _ in requests]
    result = asyncio.run(agent.ainvoke(
        Command(resume={"decisions": decisions}),
        config=run_config,
    ))
    result["messages"][-1].pretty_print()


## 10. Durable state across CLI sessions

Create a new thread with no prior conversation history and ask for the event. If the event appears, SQLite—not conversational memory—is carrying the family fact across sessions.

![New session, same family calendar](./images/10_img.png)


In [ ]:
verify_config = {
    "configurable": {"thread_id": f"notebook-verify-{uuid4()}"},
    "recursion_limit": 30,
}
verify_result = asyncio.run(agent.ainvoke(
    {"messages": [{"role": "user", "content": f"Show activities on {demo_day.isoformat()} for family-1"}]},
    config=verify_config,
))
verify_result["messages"][-1].pretty_print()


## 11. Family outing research with You.com

The Family Outing Agent checks calendar and school commitments through one
controlled `research_family_outings` wrapper. The hosted connection discovers
seven You.com tools, but the specialist receives only the wrapper, which uses
`you-search` and `you-contents`. The model explains cited results without
inventing hours, prices, tickets, travel time, or suitability. Search results
are proposals and never become calendar events without a separate approved
calendar action.

In [ ]:
outing_prompt = (
    "Find three science or outdoor activities near San Jose for family-1 this "
    "weekend. Check our family and school calendars first, stay within a "
    "45-minute drive, preserve source URLs, and do not add anything to the calendar."
)
print("Try this after adding YDC_API_KEY to .env:\n", outing_prompt)


## 12. Deep weekly coordination + school calendar

This is the workflow that makes the project visibly more agentic than a tool router. A weekly request requires several isolated specialists and shared artifacts:

1. **Intake Agent** normalizes the week and family goal.
2. **Weekly Planner** loads existing family events and Maple Grove school events.
3. **Transportation Agent** calls the deterministic availability, transportation, and candidate tools to produce three ranked options.
4. **Schedule Reviewer** audits the recommended option—not one event at a time.
5. A rejected review loops back to planning for revision and another review.
6. **Reminder Agent** drafts reminders only after review approval.
7. Proposed writes carry expected versions and are emitted together so the parent sees one grouped approval set.

The school source is the supplied Maple Grove Elementary 2026–2027 calendar. Its dates are subject to change, so this is reviewed project data rather than a live school feed.

In [ ]:
import json

school_calendar = json.loads(
    Path("data/maple_grove_elementary_2026_2027.json").read_text()
)
print(school_calendar["school"], school_calendar["school_year"])
print("Transcribed events:", len(school_calendar["events"]))

weekly_prompt = (
    "Coordinate next week for family-1, identify family, school, and transportation "
    "conflicts, generate three options for parent-1, parent-2, and vikram, recommend "
    "the best reviewed plan, and draft reminders without applying changes."
)
print("\nTry this deep workflow in the CLI:\n", weekly_prompt)


## 13. Seeded month and Kinday calendar

`tools/seed_month_demo.py` idempotently creates 21 realistic `family-1` events
from August 30 through September 29, 2026. The `frontend/` Kinday prototype
shows a snapshot of those records in month, week, day, and list views. SQLite
remains authoritative; the frontend snapshot does not update automatically
after later CLI mutations (see `frontend/README.md`).

In [ ]:
seed_command = (
    "python tools/seed_month_demo.py --database data/family_activity.db "
    "--start 2026-08-29"
)
print("Seed the demo month with:\n", seed_command)
print("Run the Kinday prototype with: cd frontend && npm install && npm run dev")


## Recap

I built a family activity system that demonstrates these core agentic patterns:

- A **Deep Agent coordinator** that routes and delegates.
- Eight **specialist subagents**, including outing research, transportation planning, and independent review.
- **Eighteen local MCP tools** plus seven discoverable hosted You.com tools; outing research is constrained behind a controlled wrapper.
- A **review/revision loop** with shared weekly-plan artifacts.
- **Human approval** before every write.
- **Deterministic Python safeguards** for past times, conflicts, versions, family scope, availability, transportation, candidate scoring, and idempotency.
- **Reminders drafted for text or WhatsApp delivery** — the drafting works today; the actual send step isn't wired up yet.
- **SQLite persistence across fresh CLI sessions**, with no need for conversational memory.
- `recursion_limit` as the safety backstop.

**Where to take it next:** add live frontend/SQLite synchronization, authenticated family membership, Google Calendar synchronization, and real text/WhatsApp delivery (e.g., via Twilio) for the drafted reminders.